# SELENE — project source notebook

Edit and save this notebook, then run `python run_notebooks.py` from the project folder.
This notebook generates `core.py`, `rasters.py`, `app.py`, and `test_app.py`. The small `run_notebooks.py` exporter remains separate so it can read the notebook without executing it.

**Do not Run All.** Source cells define separate Python modules, not one shared notebook program. The launcher exports them in document order within each module, then runs Streamlit in its own process. Kernel state and saved outputs are ignored.

Each source cell has a visible `# export: module.py` line and a matching `export:module` metadata tag. Duplicate an existing cell in the same section when adding code, preserving both. Untagged code, unknown targets, mismatched markers and syntax errors stop export before any Python files are changed.

The original four notebooks are backed up outside this deliverable. Edit only this notebook for application changes. Generated Python files are overwritten on each launch; commit them with the saved notebook if using GitHub.

Commands:
- `python run_notebooks.py --check`: validate and regenerate all four Python modules without starting the app.
- `python run_notebooks.py --test`: regenerate modules and run tests using the real source cache.
- `python run_notebooks.py --port 8503`: regenerate modules and launch Streamlit.

Stop the running server with Ctrl+C before exporting changes. Save and restart to apply notebook edits. The `.data/` cache and `.streamlit/` theme remain unchanged.


## Analysis, query planning and K-means

Export target: `core.py`. Keep imports and definitions in execution order within this section.


In [ ]:
# export: core.py
"""Deterministic data tools used by the prospecting workflow."""
import json
import re
import numpy as np
import pandas as pd
import requests
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

DEFAULT = dict(west=-60., east=60., south=-40., north=40., minimum_tio2=2., top_n=8, priority='balanced')



In [ ]:
# export: core.py
def fit_terrain(df, k=4):
    df = df.copy().reset_index(drop=True)
    if len(df)<k or len(df[['tio2','omat']].drop_duplicates())<k:
        raise ValueError('Need at least four distinct valid TiO2/OMAT samples.')
    x = StandardScaler().fit_transform(df[['tio2','omat']])
    model = KMeans(n_clusters=k, random_state=530, n_init=10).fit(x)
    df['cluster'] = model.labels_.astype(str)
    df['site_id'] = [f'L{i:05d}' for i in range(len(df))]
    df['score'] = 100*(.65*df.tio2.rank(pct=True)+.35*(-df.omat).rank(pct=True))
    sil = silhouette_score(x, model.labels_, sample_size=min(1500,len(df)), random_state=530)
    return df, float(sil)



In [ ]:
# export: core.py
def validate_plan(plan):
    if set(plan) != set(DEFAULT):
        raise ValueError('The query plan has unexpected or missing fields.')
    for key in ['west','east','south','north','minimum_tio2','top_n']:
        if isinstance(plan[key], bool) or not isinstance(plan[key], (float,int)) or not np.isfinite(plan[key]):
            raise ValueError(f'Invalid numeric value for {key}.')
    if not -180<=plan['west']<plan['east']<=180 or not -90<=plan['south']<plan['north']<=90:
        raise ValueError('Use ordered lunar bounds: west < east, south < north. Split dateline-crossing regions.')
    if not 0<=plan['minimum_tio2']<=15 or not 1<=plan['top_n']<=50 or int(plan['top_n'])!=plan['top_n']:
        raise ValueError('TiO2 must be 0–15 wt%; candidate count must be an integer from 1–50.')
    if plan['priority'] not in ['balanced','tio2','maturity']:
        raise ValueError('Unsupported ranking priority.')
    return plan



In [ ]:
# export: core.py
def plan_query(query, current, use_llm=False, model='gemma3:4b'):
    if not query.strip():
        raise ValueError('Enter a prospecting question.')
    if use_llm:
        system = ('Translate lunar prospecting requests into JSON. Only fields in the current plan are allowed. '
                  'Preserve prior values unless changed by the user. priority is balanced, tio2, or maturity. '
                  'Bounds are degrees east [-180,180] and north [-90,90]. minimum_tio2 is wt%. '
                  'For unsupported requests (helium abundance, solar wind, slope, landing safety, other elements) '
                  'return {"error":"Explain missing data"}. Do not invent measurements. Current plan: '+json.dumps(current))
        response = requests.post('http://localhost:11434/api/chat', json=dict(model=model,
            messages=[dict(role='system',content=system),dict(role='user',content=query)],
            format='json', stream=False, options={'temperature':0}), timeout=(5,90))
        response.raise_for_status()
        plan = json.loads(response.json()['message']['content'])
        if 'error' in plan:
            raise ValueError(str(plan['error']))
        return validate_plan(plan)
    q = query.lower().strip().rstrip('.?!')
    plan = dict(current)
    # Explicit, limited offline grammar: never silently ignore an arbitrary constraint.
    if q in ['find the best spots','find me the best spot for helium-3 mining','balanced']:
        plan['priority'] = 'balanced'
    elif q in ['highest tio2','show highest tio2','rank by tio2']:
        plan['priority'] = 'tio2'
    elif q in ['most mature','rank by maturity']:
        plan['priority'] = 'maturity'
    elif re.fullmatch(r'top \d+',q):
        plan['top_n'] = int(q.split()[1])
    elif re.fullmatch(r'minimum tio2 \d+(\.\d+)?',q):
        plan['minimum_tio2'] = float(q.split()[-1])
    else:
        raise ValueError('Offline mode supports: highest TiO2; most mature; top 5; minimum TiO2 4; find the best spots. Use the region controls or connect Ollama for flexible questions. Solar wind, other elements and landing safety are not available.')
    return validate_plan(plan)



In [ ]:
# export: core.py
def rank_sites(df, plan):
    validate_plan(plan)
    eligible = df[df.longitude.between(plan['west'],plan['east']) & df.latitude.between(plan['south'],plan['north']) & (df.tio2>=plan['minimum_tio2'])].copy()
    key, ascending = {'balanced':('score',False),'tio2':('tio2',False),'maturity':('omat',True)}[plan['priority']]
    return eligible.sort_values([key,'site_id'],ascending=[ascending,True]).head(int(plan['top_n']))



In [ ]:
# export: core.py
def explain(result, plan):
    if result.empty:
        return 'No samples meet these constraints. Expand the region or lower the minimum TiO2; constraints have not been relaxed automatically.'
    row = result.iloc[0]
    prefix = 'Imported raster samples: '
    return (f'{prefix}{len(result)} candidates ranked by {plan["priority"]}. '
            f'Leading sample {row.site_id} is at {row.latitude:.2f}°N, {row.longitude:.2f}°E, '
            f'with TiO2 {row.tio2:.2f} wt%, OMAT {row.omat:.3f}, and terrain group {row.cluster}. '
            'These proxies support follow-up investigation; they do not measure helium-3 or establish landing suitability.')


## Approved source downloads and sampling

Export target: `rasters.py`. Keep imports and definitions in execution order within this section.


In [ ]:
# export: rasters.py
"""Fixed publisher sources. No user-supplied files, URLs, or generated measurements."""
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import hashlib
import json
import os
import threading
import requests
import numpy as np
import pandas as pd
import rasterio
from pyproj import CRS, Transformer

SOURCE_PAGES = {
 'tio2':'https://data.lroc.im-ldi.com/lroc/view_rdr/WAC_TIO2',
 'omat':'https://astrogeology.usgs.gov/search/map/kaguya_lunar_multiband_imager_mi_derived_optical_maturity_omat_50n50s_512ppd_59mpp',
}
BASE='https://pds.lroc.im-ldi.com/data/LRO-L-LROC-5-RDR-V1.0/LROLRC_2001/DATA/SDP/WAC_TIO2/'
OMAT_URL='https://asc-pds-services.s3.us-west-2.amazonaws.com/mosaic/Lunar_MI_mineral_maps/4ppd/Lunar_Kaguya_MIMap_MineralDeconv_OpticalMaturityIndex_50N50S.tif'
TILES=[f'WAC_TIO2_E350{hem}{lon}.IMG' for hem in ['N','S'] for lon in ['0450','1350','2250','3150']]
CACHE=Path(os.environ.get('MoonScout_DATA_DIR', str(Path(__file__).parent/'.data')))
LOCK=threading.Lock()
VERSION=1



In [ ]:
# export: rasters.py
def download(url,name):
    path=CACHE/name
    receipt=path.with_suffix(path.suffix+'.json')
    if path.exists() and receipt.exists():
        meta=json.loads(receipt.read_text())
        if meta.get('source_url')==url and meta.get('bytes')==path.stat().st_size:
            return path,meta
    temporary=path.with_suffix(path.suffix+'.part')
    digest=hashlib.sha256(); count=0
    with requests.get(url,stream=True,timeout=(15,90)) as response:
        response.raise_for_status()
        with temporary.open('wb') as out:
            for chunk in response.iter_content(1024*1024):
                out.write(chunk); digest.update(chunk); count+=len(chunk)
        expected=response.headers.get('Content-Length')
        if expected and int(expected)!=count: raise ValueError(f'Incomplete source download: {name}')
        meta={'source_url':url,'resolved_url':response.url,'bytes':count,'sha256':digest.hexdigest()}
    with rasterio.open(temporary) as ds:
        if ds.count!=1 or not ds.crs: raise ValueError(f'Invalid scientific raster: {name}')
    os.replace(temporary,path)
    receipt.write_text(json.dumps(meta,indent=2))
    return path,meta



In [ ]:
# export: rasters.py
def sample(ds,lon,lat):
    crs=CRS.from_user_input(ds.crs)
    if not 1700000<crs.ellipsoid.semi_major_metre<1800000:
        raise ValueError('Publisher raster does not have a lunar CRS.')
    transformer=Transformer.from_crs(crs.geodetic_crs,crs,always_xy=True,force_over=True)
    values=np.full(len(lon),np.nan)
    for shift in [0,360,-360]:
        x,y=transformer.transform(lon+shift,lat); x=np.asarray(x); y=np.asarray(y)
        inside=(x>=ds.bounds.left)&(x<ds.bounds.right)&(y>ds.bounds.bottom)&(y<=ds.bounds.top)&np.isnan(values)
        ids=np.flatnonzero(inside)
        if len(ids):
            a=np.ma.vstack(list(ds.sample(zip(x[ids],y[ids]),indexes=1,masked=True))).astype(float)
            values[ids]=a.filled(np.nan).ravel()*ds.scales[0]+ds.offsets[0]
    return values



In [ ]:
# export: rasters.py
def load_sources(progress=lambda message: None):
    with LOCK:
        CACHE.mkdir(exist_ok=True)
        saved=CACHE/'samples.json'
        if saved.exists():
            payload=json.loads(saved.read_text())
            if payload.get('version')==VERSION:
                df=pd.DataFrame(payload['rows']); df.attrs=payload['attrs']
                return df,payload['provenance']
        progress('Downloading scientific LROC tiles and USGS OMAT. First load may take several minutes (about 1.2 GB); later visits use the local cache.')
        jobs=[(BASE+n,n) for n in TILES]+[(OMAT_URL,'omat-4ppd.tif')]
        with ThreadPoolExecutor(max_workers=4) as pool:
            files=list(pool.map(lambda job:download(*job),jobs))
        lon,lat=np.meshgrid(np.arange(-179.,180.,2.),np.arange(49.,-50.,-2.))
        df=pd.DataFrame({'longitude':lon.ravel(),'latitude':lat.ravel()})
        tio=np.full(len(df),np.nan); provenance=[]
        for (path,meta),name in zip(files[:-1],TILES):
            progress(f'Sampling {name}')
            with rasterio.open(path) as ds:
                if ds.dtypes[0]!='float32': raise ValueError('Expected scientific float32 WAC product, not browse imagery.')
                a=sample(ds,df.longitude.to_numpy(),df.latitude.to_numpy())
                good=np.isfinite(a)&(a>=0)&(a<=15)
                tio[good]=a[good]
                provenance.append({**meta,'source_page':SOURCE_PAGES['tio2'],'product':name,'units':'wt%','native_resolution':'about 400 m/pixel','scale':ds.scales[0],'offset':ds.offsets[0]})
        path,meta=files[-1]
        with rasterio.open(path) as ds:
            df['omat']=sample(ds,df.longitude.to_numpy(),df.latitude.to_numpy())
        df['tio2']=tio
        valid=np.isfinite(df[['tio2','omat']]).all(axis=1)&df.omat.between(0,1)
        df=df[valid].reset_index(drop=True)
        if len(df)<5: raise ValueError('Source products yielded too few valid overlapping samples.')
        df.attrs={'grid_longitudes':lon[0].tolist(),'grid_latitudes':lat[:,0].tolist()}
        provenance.append({**meta,'source_page':SOURCE_PAGES['omat'],'product':'Kaguya MI Optical Maturity Index 50N50S','resolution':'Publisher-provided 4 ppd overview of the 512 ppd dataset','credit':'JAXA/MoonScout; Lemelin et al. (2016)'})
        provenance.append({'sampling':'Nearest pixel at 2-degree grid centers; no missing values filled','valid_samples':len(df),'excluded_samples':int((~valid).sum()),'coverage':'180W–180E, 50S–50N','quality_filter':'TiO2 0–15 wt%; OMAT 0–1'})
        payload={'version':VERSION,'rows':df.to_dict('records'),'attrs':df.attrs,'provenance':provenance}
        tmp=saved.with_suffix('.tmp'); tmp.write_text(json.dumps(payload,allow_nan=False)); os.replace(tmp,saved)
        return df,provenance


## Streamlit and Leafmap interface

Export target: `app.py`. Keep imports and definitions in execution order within this section.


In [ ]:
# export: app.py
import json
import os
import numpy as np
import streamlit as st
import leafmap.foliumap as leafmap
import folium
from core import DEFAULT, fit_terrain, plan_query, rank_sites, explain
from rasters import load_sources, SOURCE_PAGES

st.set_page_config(page_title='MoonScout | Lunar prospecting', page_icon='🌘', layout='wide')
st.markdown('''<style>.block-container{padding-top:2rem}h1{letter-spacing:-1.8px} div[data-testid="stMetric"]{background:#151e2d;padding:12px;border-radius:12px} div[data-testid="stMetricValue"]{font-size:1.4rem} .eyebrow{color:#76e0c5;letter-spacing:3px;font-size:12px}</style>''',unsafe_allow_html=True)

# Invalidate old sessions, including data and messages from the removed demo mode.
if st.session_state.get('data_policy_version') != 3:
    st.session_state.clear()
    st.session_state.update(data_policy_version=3, data=None, silhouette=None,
                            provenance=[], plan=dict(DEFAULT), history=[])

with st.sidebar:
    st.markdown('### 🌘 MoonScout')
    st.caption('LUNAR PROSPECTING WORKSPACE')
    page = st.radio('Workspace', ['Explore','Methods & project'],label_visibility='collapsed')
    st.divider()
    st.caption('QUERY ENGINE')
    use_llm = st.toggle('Use local Ollama LLM', value=False)
    model = st.text_input('Ollama model', value=os.getenv('OLLAMA_MODEL','gemma3:4b'), disabled=not use_llm)
    st.caption('Ollama must be running on this server at localhost:11434. Offline mode uses a small, explicit command grammar.')
    st.divider()
    st.write('No dataset loaded' if st.session_state.data is None else 'Source raster data')
    if st.session_state.data is not None:
        st.caption(f'{len(st.session_state.data):,} valid samples · 4 terrain groups')

st.markdown('<div class="eyebrow">MoonScout / EXPLORATORY INTELLIGENCE</div>',unsafe_allow_html=True)
st.title('Find your next lunar lead.')
st.caption('Compare titanium abundance and optical maturity. Build a shortlist for further investigation.')
st.caption('Data sources: LROC WAC TiO₂ abundance and USGS Kaguya MI OMAT only.')

if page == 'Explore':
    if st.session_state.data is None:
        try:
            with st.status('Loading approved lunar datasets…', expanded=True) as status:
                data, provenance = load_sources(st.write)
                clustered, silhouette = fit_terrain(data)
                st.session_state.update(data=clustered, silhouette=silhouette, provenance=provenance)
                status.update(label='Source datasets loaded', state='complete', expanded=False)
            st.rerun()
        except Exception as error:
            st.error(f'The approved source datasets could not be loaded: {error}')
            st.info('No substitute data is used. Check the server connection and retry.')
            if st.button('Retry source loading'):
                st.rerun()
            st.stop()
    st.caption('Automatically loaded • LROC scientific abundance tiles + USGS OMAT 4 ppd overview • 2° sampling grid')
    plan=st.session_state.plan
    with st.expander('Region & ranking controls',expanded=False):
        with st.form('controls'):
            cols=st.columns(4)
            values={key:cols[i].number_input(key.title(),value=float(plan[key])) for i,key in enumerate(['west','east','south','north'])}
            cols=st.columns(3)
            values['minimum_tio2']=cols[0].number_input('Minimum TiO₂ (wt%)',0.,15.,float(plan['minimum_tio2']))
            values['top_n']=cols[1].number_input('Candidates',1,50,int(plan['top_n']))
            values['priority']=cols[2].selectbox('Rank by',['balanced','tio2','maturity'],index=['balanced','tio2','maturity'].index(plan['priority']))
            if st.form_submit_button('Apply constraints'):
                from core import validate_plan
                try:
                    st.session_state.plan=validate_plan(values)
                    st.rerun()
                except ValueError as error:
                    st.error(str(error))
    left,right=st.columns([.85,1.65],gap='large')
    with left:
        st.subheader('01 / Ask & refine')
        st.caption('Try “highest TiO2”, then “top 5”. Follow-ups retain the current region and filters.')
        with st.form('question',clear_on_submit=True):
            question=st.text_area('Prospecting question',placeholder='Find the best spots',height=105)
            run=st.form_submit_button('Find candidates',type='primary')
        if run:
            try:
                with st.spinner('Plan → validate → retrieve → rank → explain'):
                    new_plan=plan_query(question,st.session_state.plan,use_llm,model)
                    results=rank_sites(st.session_state.data,new_plan)
                    response=explain(results,new_plan)
                st.session_state.plan=new_plan
                st.session_state.history.append((question,response,'Ollama' if use_llm else 'Offline parser'))
            except Exception as error:
                st.error(f'Query stopped; previous results retained. {error}')
        for question,response,engine in st.session_state.history[-4:]:
            with st.chat_message('user'): st.write(question)
            with st.chat_message('assistant'):
                st.write(response)
                st.caption(engine)
        if st.button('Clear conversation'):
            st.session_state.history=[]
            st.rerun()
        with st.expander('Workflow trace'):
            st.write('1. Parse request into a constrained plan.\n2. Validate bounds and available fields.\n3. Retrieve matching samples.\n4. Rank by proxies and attach fitted terrain groups.\n5. Explain using exact retrieved values.')
            st.json(st.session_state.plan)
    with right:
        result=rank_sites(st.session_state.data,st.session_state.plan)
        st.subheader('02 / Survey the candidates')
        a,b,c=st.columns(3)
        a.metric('Shortlisted',len(result))
        b.metric('Top TiO₂',f'{result.tio2.max():.2f} wt%' if len(result) else '—')
        c.metric('Terrain groups',result.cluster.nunique())
        layer=st.selectbox('Surface layer',['TiO₂ abundance','Optical maturity','Terrain groups'])
        # Simple CRS maps lunar longitude/latitude directly; Earth distance tools are disabled.
        m=leafmap.Map(center=[0,0],zoom=1,tiles=None,crs='Simple',draw_control=False,
                      measure_control=False,fullscreen_control=True,latlon_control=False,
                      search_control=False,scale_control=False)
        for key in list(m._children):
            if isinstance(m._children[key],folium.raster_layers.TileLayer): del m._children[key]
        data=st.session_state.data
        region=data[data.longitude.between(st.session_state.plan['west'],st.session_state.plan['east']) & data.latitude.between(st.session_state.plan['south'],st.session_state.plan['north'])]
        field={'TiO₂ abundance':'tio2','Optical maturity':'omat','Terrain groups':'cluster'}[layer]
        colors=['#70e1c4','#f5b86b','#9898f0','#eb83ae']
        # Reconstruct the imported regular grid as a small RGBA raster overlay.
        if len(region):
            p=st.session_state.plan
            xs=np.sort(np.asarray(data.attrs.get('grid_longitudes',data.longitude.unique())))
            ys=np.sort(np.asarray(data.attrs.get('grid_latitudes',data.latitude.unique())))[::-1]
            xs=xs[(xs>=p['west'])&(xs<=p['east'])]; ys=ys[(ys>=p['south'])&(ys<=p['north'])]
            if len(xs)>1 and len(ys)>1:
                grid=np.zeros((len(ys),len(xs),4),dtype=np.uint8)
                xidx={x:i for i,x in enumerate(xs)}; yidx={y:i for i,y in enumerate(ys)}
                low,high=region[field].astype(float).min(),region[field].astype(float).max()
                for row in region.itertuples():
                    value=float(getattr(row,field)); t=(value-low)/(high-low or 1)
                    color=colors[int(row.cluster)] if field=='cluster' else None
                    rgb=[int(color[i:i+2],16) for i in (1,3,5)] if color else [int(35+210*t),int(65+130*t),int(90+30*t)]
                    grid[yidx[row.latitude],xidx[row.longitude]]=[*rgb,210]
                dx=(xs[1]-xs[0])/2; dy=abs(ys[1]-ys[0])/2
                folium.raster_layers.ImageOverlay(grid,bounds=[[ys[-1]-dy,xs[0]-dx],[ys[0]+dy,xs[-1]+dx]],name=layer,origin='upper').add_to(m)
        for rank,row in enumerate(result.itertuples(),1):
            popup=f'#{rank} · {row.site_id}<br>{row.latitude:.2f}°N, {row.longitude:.2f}°E<br>TiO₂ {row.tio2:.2f} wt%<br>OMAT {row.omat:.3f}<br>Terrain {row.cluster}'
            folium.Marker([row.latitude,row.longitude],tooltip=f'#{rank} {row.site_id}',popup=popup,
                icon=folium.DivIcon(html=f'<div style="background:#76e0c5;color:#101722;border:2px solid white;border-radius:50%;width:25px;height:25px;text-align:center;font-weight:bold">{rank}</div>')).add_to(m)
        p=st.session_state.plan
        m.fit_bounds([[p['south'],p['west']],[p['north'],p['east']]])
        m.to_streamlit(height=410)
        st.caption('Lunar longitude/latitude · sampled raster preview, not full-resolution imagery. Continuous layers: dark = low, gold = high. Terrain groups: mint 0 · gold 1 · violet 2 · pink 3.')
        if len(region) and field!='cluster':
            st.caption(f'Visible color range: {region[field].min():.3f} to {region[field].max():.3f} '+('wt%' if field=='tio2' else 'OMAT'))
        if result.empty: st.info(explain(result,p))
        else:
            st.dataframe(result[['site_id','latitude','longitude','tio2','omat','cluster','score']].round(3),hide_index=True,width='stretch')
            st.download_button('Download shortlist (CSV)',result.assign(tio2_source=SOURCE_PAGES['tio2'],omat_source=SOURCE_PAGES['omat']).to_csv(index=False),'lunar-shortlist.csv','text/csv')
            st.download_button('Download query & provenance',json.dumps({'plan':p,'provenance':st.session_state.provenance},indent=2,default=str),'provenance.json','application/json')

else:
    st.subheader('A prospecting aid, with evidence in view')
    st.write('Designed for exploratory analysts comparing potential lunar prospecting regions. A successful interaction produces a traceable shortlist, not a mining or landing authorization.')
    st.markdown('### How the pieces fit')
    st.code('Question → LLM JSON plan → validation → raster sample retrieval\n                                           ↓\nPublisher rasters → common lunar grid → standardization → K-means groups\n                                           ↓\n                     proxy ranking → Leafmap + table + explanation')
    st.markdown('### Traditional ML')
    st.write('K-means groups samples using standardized TiO₂ and OMAT (4 groups, 10 initializations, seed 530). Latitude and longitude are excluded so groups represent similar proxy values rather than proximity. Group labels appear on the map and shortlist; the group summary supports comparison. Labels are arbitrary and refitted when the source sample set changes.')
    if st.session_state.silhouette is not None:
        st.metric('Sampled silhouette score',f'{st.session_state.silhouette:.3f}')
    else:
        st.caption('Model results appear after automatic source loading on Explore.')
    st.caption('Cluster separation only; this is not evidence of helium-3 prediction accuracy. Spatially correlated sampling and choice of region affect the result.')
    if st.session_state.data is not None:
        st.dataframe(st.session_state.data.groupby('cluster').agg(samples=('site_id','count'),mean_tio2=('tio2','mean'),mean_omat=('omat','mean')).round(3))
    st.markdown('### Ranking and limitations')
    st.write('Balanced score = 100 × (0.65 × TiO₂ percentile + 0.35 × inverse-OMAT percentile) within the loaded source dataset. This is an explicit exploratory heuristic, not a trained helium-3 model. Lower OMAT is treated as greater optical maturity, not a direct solar-wind exposure measurement. Pure TiO₂ and maturity rankings are also available. No helium-3 ground truth, slope, landing safety, other elements, extraction economics or direct solar-wind data are included.')
    st.write('WAC abundance values below 2 wt% are below the product detection limit; values above 15 wt% are excluded. OMAT outside 0–1 is excluded as a prototype quality check. Verify product-specific calibration and masks. Grid sampling can miss small high-value regions; nearby shortlist rows are not independent deposits.')
    st.markdown('### Product decisions')
    st.write('1. Keep conversation beside the map and table, matching the scoping sketch. Explicit region controls and a visible parsed plan make constraints inspectable.\n\n2. Load only fixed publisher sources and cache successful downloads: source failures show an error instead of substitute data. The workflow stops on unsupported questions instead of silently dropping constraints. These are implementation decisions; user testing remains to be conducted.')
    with st.expander('Loaded source provenance'):
        st.json(st.session_state.provenance)
    st.caption('OMAT uses the publisher-provided 4 ppd scientific overview of the linked 512 ppd product. The map samples every 2 degrees and cannot identify local maxima at full raster resolution. Credit: JAXA/MoonScout; Lemelin et al. (2016).')
    st.markdown('### Sources & setup')
    st.markdown('- [LROC WAC TiO₂ product and calibration](https://data.lroc.im-ldi.com/lroc/view_rdr/WAC_TIO2)\n- [USGS Kaguya OMAT product](https://astrogeology.usgs.gov/search/map/kaguya_lunar_multiband_imager_mi_derived_optical_maturity_omat_50n50s_512ppd_59mpp)\n- [Leafmap Streamlit integration](https://leafmap.org/foliumap/)\n- [Ollama chat API](https://docs.ollama.com/api/chat)')
    st.info('LLM integration is implemented but requires a running Ollama model. Offline mode is not an LLM. A recorded walkthrough, representative LLM benchmark and real-product calibration validation remain follow-up work.')


## Application tests

Export target: `test_app.py`. Keep imports and definitions in execution order within this section.


In [ ]:
# export: test_app.py
"""Tests use publisher data or a network failure, never generated measurements."""
import unittest
from unittest.mock import patch
from core import DEFAULT, plan_query
from rasters import load_sources
from streamlit.testing.v1 import AppTest



In [ ]:
# export: test_app.py
class AutomaticSourceTests(unittest.TestCase):
    def test_followup(self):
        p=plan_query('top 5',plan_query('highest TiO2',DEFAULT))
        self.assertEqual(p['top_n'],5)
        self.assertEqual(p['priority'],'tio2')
    def test_sources_and_cache(self):
        data,provenance=load_sources()
        self.assertGreater(len(data),100)
        self.assertTrue(data.tio2.between(0,15).all())
        self.assertEqual(len([p for p in provenance if 'sha256' in p]),9)
        again,_=load_sources()
        self.assertTrue(data.equals(again))
    def test_automatic_app(self):
        at=AppTest.from_file(str(__import__('pathlib').Path(__file__).with_name('app.py')),default_timeout=120).run()
        self.assertFalse(at.exception)
        self.assertIsNotNone(at.session_state['data'])
        self.assertEqual(at.sidebar.radio[0].options,['Explore','Methods & project'])
        self.assertEqual(len(at.get('file_uploader')),0)
        at.text_area[0].set_value('highest TiO2')
        [b for b in at.button if b.label=='Find candidates'][0].click()
        at.run()
        self.assertFalse(at.exception)
        self.assertEqual(at.session_state['plan']['priority'],'tio2')
        at.sidebar.radio[0].set_value('Methods & project').run()
        self.assertFalse(at.exception)
    def test_source_failure_has_no_fallback(self):
        with patch('rasters.load_sources',side_effect=ConnectionError('Source unavailable')):
            at=AppTest.from_file(str(__import__('pathlib').Path(__file__).with_name('app.py')),default_timeout=120).run()
        self.assertFalse(at.exception)
        self.assertIsNone(at.session_state['data'])
        self.assertEqual(len(at.dataframe),0)
        self.assertTrue(any('could not be loaded' in e.value for e in at.error))
        self.assertTrue(any(b.label=='Retry source loading' for b in at.button))



In [ ]:
# export: test_app.py
if __name__=='__main__': unittest.main()


## Practical limits

- This notebook is an editing and export document, not a Run All analysis notebook. Module-level imports, `__file__`, Streamlit session state and test execution need the generated file contexts.
- Unsaved edits are not exported; a running server does not automatically re-export notebook changes.
- Generated `.py` edits are overwritten. Keep logic in this notebook; the launcher is separate infrastructure.
- Reordering cells changes source order. New circular imports between modules can still fail at runtime even if syntax validation passes.
- One notebook concentrates Git merge conflicts. Clear outputs before committing; do not save secrets in cells or outputs.
- The whole notebook is loaded and validated before export. Replacement is atomic per file, not an all-files transaction; interrupted exports can leave mixed versions. Rerun `--check` before restarting.
- The data cache is outside this notebook. The first load still downloads the approved datasets; no synthetic data is introduced.
